In [1]:
import requests
import os
import numpy as np
import pandas as pd

## From Vergil's code

In [4]:
def get_neuron_model_morphometrics(nmldb_id):

    nmldb_url = 'http://neuroml-db.org/api/morphometrics?id='

    neuron_url = nmldb_url + nmldb_id

    nmldb_morpho_response = requests.get(neuron_url, verify=False)

    return nmldb_morpho_response.json()

In [5]:
NMLDB_ID = 'NMLCL000073'

# retrieve model height (to place pseudo-electrode points)
example_neuron_morpho = get_neuron_model_morphometrics(NMLDB_ID)
for metric_dict in example_neuron_morpho:
    if metric_dict['Metric_ID'] == 'Height':
        model_height = metric_dict['Maximum']
    if metric_dict['Metric_ID'] == 'Width':
        model_width = metric_dict['Maximum']
    if metric_dict['Metric_ID'] == 'Depth':
        model_depth = metric_dict['Maximum']

/home/kedoxey/.conda/envs/python3_9-NEW/lib/python3.9/site-packages/pip_system_certs/wrapt_requests.py:71: UserWarning: Failed to patch SSL settings for unverified requests (unsupported version of urllib3?)
This may lead to errors when urllib3 tries to modify verify_mode.
Please report an issue at https://gitlab.com/alelec/pip-system-certs with your
python version included in the description

  warnings.warn(
/home/kedoxey/.conda/envs/python3_9-NEW/lib/python3.9/site-packages/urllib3/connectionpool.py:1045: InsecureRequestWarning: Unverified HTTPS request is being made to host 'neuroml-db.org'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/1.26.x/advanced-usage.html#ssl-warnings
  warnings.warn(


In [6]:
# define recording electrode geometry (get more precise)
channel_spacing = 50. # (microns)
buffer_dim = 50. # (microns)

# TODO: Should be defined based on y minimum with soma placed at origin
y_shift = -500 # (microns)

# get dimensions divisible by channel_spacing
# (DEPRECATED)
x_dim =  channel_spacing*np.floor((model_width+buffer_dim)/channel_spacing)+channel_spacing
y_dim =  channel_spacing*np.floor((model_height+buffer_dim)/channel_spacing)+channel_spacing
z_dim =  channel_spacing*np.floor((model_depth+buffer_dim)/channel_spacing)+channel_spacing

# make x-z plane square
if x_dim>z_dim:
    z_dim = x_dim
else:
    x_dim = z_dim

In [7]:
sampling_dist = 'fixed-normal' # 'fixed-normal' or 'adjusted-lognormal' or 'adjusted-uniform'

#### Define recording electrodes geometry   
num_probes = 100

# sample radii from normal distribution
print('Adjusting sample space: %s'%sampling_dist)
if sampling_dist == 'fixed-normal':
    r_pos = np.random.normal(loc=20,scale=5,size=num_probes)
    
    min_radius = 10.
    r_pos = [r if r>min_radius else min_radius for r in r_pos] # force lower bound of 10.
    

Adjusting sample space: fixed-normal


In [9]:
channel_lb = 32
channel_ub = 32

grid_spacing = 10

# sample angle from uniform distribution
theta_pos = np.random.uniform(-np.pi,np.pi,size=num_probes)

all_xs = np.multiply(r_pos,np.cos(theta_pos))
all_zs = np.multiply(r_pos,np.sin(theta_pos))

all_ys = np.random.uniform(low=-0.5*grid_spacing,
                               high=0.5*grid_spacing,
                              size=num_probes)

rec_probes = []
probe_list = []

lb = int(channel_lb*grid_spacing)
ub = int(channel_ub*grid_spacing)

# iterate over probe details
for y_center, x_center, z_center in zip(all_ys,all_xs,all_zs):

    y_lower = np.arange(y_center-lb,y_center,grid_spacing)
    y_upper = np.arange(y_center,y_center+ub,grid_spacing)

    y_channels = list(y_lower) + list(y_upper)

    probe = [[x_center,yi,z_center] for yi in y_channels]
    probe_list.append(probe)

    rec_probes += probe

rec_electrode = rec_probes

num_channels, _ = np.shape(rec_electrode)

In [10]:
temp = 5

### For each location out of 100, there is a linear probe with 64 electrodes that are 10 um apart along y-axis.

In [ ]:
r_pos = np.arange(10,125,5) # 23 r-values
thetas = np.random.uniform(-np.pi,np.pi,size=10)

all_xs = []
all_zs = []
all_r_thetas = []

for ri in r_pos:

    xi = np.multiply(ri,np.cos(thetas))
    zi = np.multiply(ri,np.sin(thetas))
    
    r_thetas = [(ri, theta) for theta in thetas]

    all_xs+=list(xi)
    all_zs+=list(zi)
    all_r_thetas.extend(r_thetas)

all_xs = np.array(all_xs)
all_zs = np.array(all_zs)      
    
all_ys = np.arange(-10,15,5) # can compute SNR/amplitude decay along the Y-axis

rec_probes = []
probe_list = []
r_thetas_list = []

for xi, zi, r_thetas in zip(all_xs,all_zs, all_r_thetas):
    probe = [[xi,yi,zi] for yi in all_ys]
    probe_list.append(probe)

    r_thetas_list.extend([r_thetas for yi in all_ys])

    rec_probes += probe

            
rec_electrode = rec_probes

In [9]:
len(r_thetas_list), len(rec_probes)

(230, 1150)